In [ ]:
import json
with open('../data/processed/vectorization_time.json', 'r') as f:
    v_times = json.load(f)

# Теперь v_times['bert'] вернет точное время векторизации


In [29]:
import time
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
# from src.data_loader import get_methods

# Список для хранения расширенной статистики
detailed_results = []
# Список методов для сопоставления с именами файлов
methods = [
    ("Binary", "binary"),
    ("Bag of Words", "bow"),
    ("TF-IDF Standard", "tfidf_std"),
    ("TF-IDF Bigrams", "tfidf_bigrams"),
    ("Word2Vec", "w2v"),
    ("Doc2Vec", "d2v"),
    ("BERT", "bert")
]
y_train = np.load(f'../data/processed/y_train.npy')
y_test = np.load(f'../data/processed/y_test.npy')
for name, key_word in methods:
    print(f"Work with method {name}")
    # 1. Загрузка (считаем частью работы с данными)
    xt = np.load(f'../data/processed/train_{key_word}_vectors.npy')
    xv = np.load(f'../data/processed/test_{key_word}_vectors.npy')
    
    # 2. Замер времени Обучения+Предсказания
    start_train = time.time()
    clf = LogisticRegression(max_iter=1000)
    clf.fit(xt, y_train)
    
    # 3. Предсказание и оценка
    y_pred = clf.predict(xv)
    acc = accuracy_score(y_test, y_pred)
    
    train_time = round(time.time() - start_train, 2)
    
    # Добавляем данные в список
    detailed_results.append({
        "Method": name,
        "Vectorization Time": v_times[key_word],
        "Training Time": train_time,
        "Total Time": round(v_times[key_word] + train_time, 2),
        "Accuracy": round(acc, 4)
    })

# Создаем красивый отчет
df_detailed = pd.DataFrame(detailed_results).sort_values(by = "Accuracy", ascending = False)
print(df_detailed)

Work with method Binary
Work with method Bag of Words
Work with method TF-IDF Standard
Work with method TF-IDF Bigrams
Work with method Word2Vec
Work with method Doc2Vec
Work with method BERT
            Method  Vectorization Time  Training Time  Total Time  Accuracy
6             BERT              568.09          91.59      659.68    0.6989
2  TF-IDF Standard                1.64         163.82      165.46    0.6907
3   TF-IDF Bigrams                4.93         182.13      187.06    0.6862
0           Binary                1.78          69.01       70.79    0.6427
1     Bag of Words                1.92         172.86      174.78    0.6371
5          Doc2Vec               78.44           2.32       80.76    0.5324
4         Word2Vec               11.20           7.74       18.94    0.3660


In [27]:
import os
os.makedirs("../results/tables/", exist_ok=True)

df_detailed.to_csv("../results/tables/classification_compare_compact.csv", index = False)
print("Результаты сохранены в ../results/tables/classification_compare_compact.csv")

Результаты сохранены в ../results/classification_compare_compact.csv
